# Evaluación del sistema de generación aumentada por recuperación (RAG)

Este notebook presenta la preparación y evaluación del sistema RAG desarrollado para el asistente de contratación pública. Se analiza la organización de las fuentes documentales, la calidad de la recuperación de información, su integración con el chat y las diferencias entre las respuestas generadas con y sin recuperación documental.


## Preparación del corpus

Se preparan los documentos de referencia relacionados con la contratación pública y la metodología de detección de anomalías. Se extrae y organiza su contenido en fragmentos para facilitar su posterior recuperación, conservando los archivos originales sin modificaciones.


## 1. Configuración

Se establecen las rutas de los documentos, los módulos necesarios y los parámetros para preparar el corpus documental.
Si los módulos se importan desde src/ de este repositorio. Los PDF deben estar en data/corpus/originales/.
Nota. 
Si falta `pypdf` o `pyarrow`:
```text
pip install --user pypdf pyarrow
```


In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

aqui = Path.cwd().resolve()
RAIZ = next(
    (
        c
        for c in (aqui, *aqui.parents)
        if (c / "src" / "rag").is_dir() and (c / "notebooks").is_dir()
    ),
    None,
)
if RAIZ is None:
    raise FileNotFoundError("Ejecute el notebook desde el repositorio secop-tfm.")
sys.path.insert(0, str(RAIZ / "src"))

from rag.preparar import preparar_corpus, verificar_dependencias
from rag.rutas import ORIGEN_WINDOWS, raiz_repo, ruta_originales, ruta_salidas_rag, sincronizar_originales

REEJECUTAR = True
SALIDAS = ruta_salidas_rag()
ORIGINALES = ruta_originales()

print("Raíz:", raiz_repo())
print("Originales:", ORIGINALES)
print("Salidas:", SALIDAS)
print("Origen Windows:", ORIGEN_WINDOWS, "existe:", ORIGEN_WINDOWS.exists())
print("REEJECUTAR:", REEJECUTAR)


Raíz: /home/jovyan/work
Originales: /home/jovyan/work/data/corpus/originales
Salidas: /home/jovyan/work/spark-apps/notebooks/salidas/rag
Origen Windows: C:\Repositorios\tfm\corpus existe: False
REEJECUTAR: True


## 2. Verificación de dependencias

Se comprueba la disponibilidad de las bibliotecas necesarias para extraer, procesar y almacenar el contenido de los documentos.

In [2]:
deps = verificar_dependencias()
display(pd.DataFrame([{"paquete": k, "estado": v} for k, v in deps.items()]))
faltan = [k for k, v in deps.items() if v != "ok"]
if faltan:
    raise ImportError(
        "Instalar en el kernel de Jupyter: pip install --user " + " ".join(faltan)
    )


,paquete,estado
0,pandas,ok
1,pypdf,ok
2,pyarrow,ok


## 3. Preparación de documentos originales

Se verifica la disponibilidad de los documentos y se organizan las fuentes utilizadas para construir el corpus. Los archivos originales se conservan sin modificaciones y se identifican aquellos que no serán procesados


In [3]:
sync = sincronizar_originales()
print("Origen visible:", sync["origen_existe"])
print("Copiados en esta corrida:", sync["copiados"] or "(ninguno; el origen no está en este entorno)")
print("Ausentes en origen:", sync["ausentes_en_origen"] or "—")
print("En destino:")
for n in sync["en_destino"]:
    print(" -", n)


Origen visible: False
Copiados en esta corrida: (ninguno; el origen no está en este entorno)
Ausentes en origen: —
En destino:
 - CCE-MINIMA-CUANTIA-2024.pdf
 - DiccionarioDeDatosAbiertos.pdf
 - Guia-Proceso-Sancionatorio-Vs2-OrigFirmado-2.pdf
 - cce-eicp-ma-04._manual_requisitos_habilitantes_v3_29-09-2023.pdf
 - expresiones_ciudadanas_simuladas_TFM.xlsx
 - ficha_c1_isolation_forest.md


## 4. Extracción y fragmentación del contenido

Se extrae el texto de los documentos y se aplican ajustes de limpieza para mejorar su procesamiento. Posteriormente, el contenido se divide en fragmentos con identificadores que permiten conservar su relación con los documentos originales.


In [4]:
res = preparar_corpus(reescribir=REEJECUTAR)
print("UTC:", res["generado_utc"])
print("Documentos incluidos:", res["n_documentos_incluidos"])
print("Páginas:", res["n_paginas"], "| Fragmentos:", res["n_fragmentos"])
print("Salidas:", res["salidas"])
display(res["resumen"])


UTC: 2026-11-02T17:26:06Z
Documentos incluidos: 4
Páginas: 69 | Fragmentos: 254
Salidas: /home/jovyan/work/spark-apps/notebooks/salidas/rag


,doc_id,archivo,titulo,tipo,incluir,error,n_paginas,n_paginas_ok,n_paginas_vacias,n_paginas_cortas,n_paginas_sospechosas,n_fragmentos,n_caracteres,sha256,bytes
0,cce-minima-cuantia-2024,CCE-MINIMA-CUANTIA-2024.pdf,Manual de mínima cuantía (Colombia Compra Efic...,pdf,True,None,19,19,0,0,0,60,34049,4ab5daa8434df53f7cf50a27914e96557cb01254a4dcae...,560750
1,cce-requisitos-habilitantes-v3-2023,cce-eicp-ma-04._manual_requisitos_habilitantes...,Manual de requisitos habilitantes v3 (29 de se...,pdf,True,None,46,46,0,0,0,174,97563,2912589d9c02f395337bc89d512efe25f525e596c7d69d...,1152821
2,diccionario-datos-abiertos-secop,DiccionarioDeDatosAbiertos.pdf,Diccionario de datos abiertos SECOP,pdf,True,None,3,3,0,0,0,11,6955,2732c404833963fc39ab8490bcda1e8a84936db5361c00...,217535
3,ficha-c1-if-v1,ficha_c1_isolation_forest.md,Ficha metodológica C1 + Isolation Forest (c1_i...,markdown,True,None,1,1,0,0,0,9,3792,f693885d8f491f8b832da293901662a8dce93b4057e8d5...,3907


## 5. Validación de la calidad del contenido

Se revisa la calidad del texto extraído para identificar páginas vacías, incompletas o con posibles problemas de extracción. También se verifican los documentos excluidos del procesamiento.

In [5]:
pag = res["paginas"]
frag = res["fragmentos"]

calidad = (
    pag.groupby(["doc_id", "calidad"])
    .size()
    .rename("n_paginas")
    .reset_index()
    .pivot_table(index="doc_id", columns="calidad", values="n_paginas", fill_value=0)
    .reset_index()
)
display(calidad)

problemas = pag[pag["calidad"].isin(["vacia", "corta", "sospechosa"])][
    ["doc_id", "pagina", "n_caracteres", "calidad", "vista_previa"]
]
print("Páginas con alerta:", len(problemas))
display(problemas.head(30))

print("Excluidos (conservados, no extraídos):")
display(pd.DataFrame(res["excluidos"]))


calidad,doc_id,ok
0,cce-minima-cuantia-2024,19
1,cce-requisitos-habilitantes-v3-2023,46
2,diccionario-datos-abiertos-secop,3
3,ficha-c1-if-v1,1


Páginas con alerta: 0


,doc_id,pagina,n_caracteres,calidad,vista_previa


Excluidos (conservados, no extraídos):


,doc_id,archivo,incluir,motivo,conservado,bytes,procesado
0,guia-proceso-sancionatorio-vs2,Guia-Proceso-Sancionatorio-Vs2-OrigFirmado-2.pdf,False,"Excluido de la v1: proceso sancionatorio, arch...",True,25765970,False


## 6. Revisión de fragmentos

Se presentan ejemplos de los fragmentos generados para comprobar su contenido, identificación y relación con las fuentes documentales.

In [6]:
cols = [
    "fragment_id", "doc_id", "titulo", "seccion",
    "pagina_inicio", "pagina_fin", "n_caracteres", "calidad", "version", "fuente",
]
muestra = (
    frag.sort_values(["doc_id", "orden"])
    .groupby("doc_id", as_index=False)
    .head(2)
)
display(muestra[cols])

print("--- texto (primeros 500 caracteres de cada ejemplo) ---")
for _, row in muestra.iterrows():
    print("\n#", row["fragment_id"])
    print("sección:", row["seccion"])
    print((row["texto"] or "")[:500])
    print("---")


,fragment_id,doc_id,titulo,seccion,pagina_inicio,pagina_fin,n_caracteres,calidad,version,fuente
0,cce-minima-cuantia-2024-p001-001-fce98d2b,cce-minima-cuantia-2024,Manual de mínima cuantía (Colombia Compra Efic...,MANUAL DE LA MODALIDAD DE SELECCIÓN DE MÍNIMA ...,1,2,1045,ok,2024,PDF local del corpus del TFM
1,cce-minima-cuantia-2024-p002-002-256e4c66,cce-minima-cuantia-2024,Manual de mínima cuantía (Colombia Compra Efic...,TABLA DE CONTENIDO,2,2,239,ok,2024,PDF local del corpus del TFM
60,cce-requisitos-habilitantes-v3-2023-p001-001-0...,cce-requisitos-habilitantes-v3-2023,Manual de requisitos habilitantes v3 (29 de se...,MANUAL PARA DETERMINAR Y VERIFICAR LOS REQUISITOS,1,2,413,ok,v3,PDF local del corpus del TFM
61,cce-requisitos-habilitantes-v3-2023-p002-002-8...,cce-requisitos-habilitantes-v3-2023,Manual de requisitos habilitantes v3 (29 de se...,CONTENIDO,2,2,874,ok,v3,PDF local del corpus del TFM
234,diccionario-datos-abiertos-secop-p001-001-40e2...,diccionario-datos-abiertos-secop,Diccionario de datos abiertos SECOP,FORMATO DICCIONARIO DE DATOS ABIERTOS,1,1,580,ok,01,PDF local del corpus del TFM
235,diccionario-datos-abiertos-secop-p001-002-9839...,diccionario-datos-abiertos-secop,Diccionario de datos abiertos SECOP,ID Nombre Campo Tipo Descripción,1,1,881,ok,01,PDF local del corpus del TFM
245,ficha-c1-if-v1-p001-001-4ef2a9e5,ficha-c1-if-v1,Ficha metodológica C1 + Isolation Forest (c1_i...,Ficha metodológica C1 + Isolation Forest (`c1_...,1,1,409,ok,c1_if_v1,docs/10_experimentos_anomalias_contratos.md; d...
246,ficha-c1-if-v1-p001-002-63fbcd8d,ficha-c1-if-v1,Ficha metodológica C1 + Isolation Forest (c1_i...,Qué mide el modelo,1,1,290,ok,c1_if_v1,docs/10_experimentos_anomalias_contratos.md; d...


--- texto (primeros 500 caracteres de cada ejemplo) ---

# cce-minima-cuantia-2024-p001-001-fce98d2b
sección: MANUAL DE LA MODALIDAD DE SELECCIÓN DE MÍNIMA CUANTÍA
MANUAL DE LA MODALIDAD DE SELECCIÓN DE MÍNIMA CUANTÍA

AGENCIA NACIONAL DE CONTRATACIÓN PÚBLICA
-COLOMBIA COMPRA EFICIENTE-
2024

MANUAL DE LA MODALIDAD DE SELECCIÓN DE MÍNIMA CUANTÍA

TABLA DE CONTENIDO
INTRODUCCIÓN 4
OBJETIVO 4
ALCANCE 4
SELECCIÓN DE MÍNIMA CUANTÍA 5
1. ¿Cómo se determina el valor de la Mínima Cuantía? 5
2. Documentos del Proceso Modalidad de Mínima Cuantía 6
2.1. Estudios previos 6
2.2. Invitación a participar 7
2.3. Cronograma del proceso 7
3. Solicitud para subsanar documen
---

# cce-minima-cuantia-2024-p002-002-256e4c66
sección: TABLA DE CONTENIDO
14
11. Plazos del Proceso de Contratación de Mínima Cuantía 15
12. Concurrencia de dos modalidades de selección 15
12.1. Concurrencia de contratación directa y mínima cuantía 15
12.2. Concurrencia de concurso de méritos y mínima cuantía 17
---

# cce-requisi

## 7. Verificación de archivos generados

Se comprueba la existencia de los archivos resultantes y la consistencia de los identificadores utilizados para organizar los fragmentos del corpus.

In [7]:
esperados = [
    "fragmentos.parquet",
    "paginas.parquet",
    "calidad_paginas.csv",
    "indice_fragmentos.csv",
    "resumen_documentos.csv",
    "informe_preparacion.json",
]
for nombre in esperados:
    p = SALIDAS / nombre
    print(nombre, "OK" if p.is_file() else "FALTA", p.stat().st_size if p.is_file() else "")

assert len(frag) == res["n_fragmentos"]
assert frag["fragment_id"].is_unique
assert set(frag["doc_id"]) <= {
    "cce-minima-cuantia-2024",
    "cce-requisitos-habilitantes-v3-2023",
    "diccionario-datos-abiertos-secop",
    "ficha-c1-if-v1",
}
assert "guia-proceso-sancionatorio-vs2" not in set(frag["doc_id"])
print("Comprobaciones de identificadores: ok")


fragmentos.parquet OK 96956
paginas.parquet OK 88758
calidad_paginas.csv OK 24253
indice_fragmentos.csv OK 122444
resumen_documentos.csv OK 1046
informe_preparacion.json OK 7658
Comprobaciones de identificadores: ok


## 8. Procedimiento

Para repetir la preparación del corpus, incluyendo la ubicación de los documentos, las dependencias y la ejecución de las etapas de procesamiento, se deberá ejecutar los siguientes pasos:

1. Colocar los PDF y el Excel de expresiones en data/corpus/originales/ de este repositorio.
2. Abrir Jupyter con el directorio de trabajo en notebooks/ o en la raíz del repositorio.
3. Instalar pypdf y pyarrow en el kernel si faltan.
4. Ejecutar todas las celdas. `REEJECUTAR = True` reescribe Parquet y CSV en notebooks/salidas/rag/.
5. Revisar páginas vacías o sospechosas antes de la evaluación de recuperación.

Los PDF de data/corpus/originales/ no se sobrescriben: solo se leen.


## Evaluación de la recuperación documental

Se evalúan diferentes estrategias para recuperar fragmentos relevantes a partir de preguntas relacionadas con la contratación pública y la metodología de detección de anomalías. El objetivo es determinar qué estrategia ofrece una mejor cobertura de la información esperada.

Reutiliza los 234 embeddings de `text-embedding-3-small` (dimensión 1536). No reescribe el Parquet del corpus.

Compara tres estrategias sobre aproximadamente 40 preguntas:

- **A.** similitud coseno top-5
- **B.** coseno top-5 más fragmentos adyacentes del mismo documento (`orden ± 1`)
- **C.** fusión RRF de coseno y BM25 (léxico), top-5, mismos embeddings

El gold y la cobertura esperada (`suficiente`, `parcial`, `no_cubierta`) se fijaron con el texto documental **antes** de ranking. No se usa GPT-4o-mini. No se usan los 26 EC reservados.


In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

aqui = Path.cwd().resolve()
RAIZ = next(
    (
        c
        for c in (aqui, *aqui.parents)
        if (c / "src" / "rag").is_dir() and (c / "notebooks").is_dir()
    ),
    None,
)
if RAIZ is None:
    raise FileNotFoundError("Ejecute el notebook desde el repositorio secop-tfm.")
sys.path.insert(0, str(RAIZ / "src"))

from rag.embeddings import clave_disponible
from rag.evaluar_ampliada import evaluar_ampliada
from rag.indexar import cargar_indice_embeddings
from rag.pruebas_ampliadas import PREGUNTAS_AMPLIADAS, resumen_conjunto
from rag.rutas import ruta_salidas_rag

SALIDAS = ruta_salidas_rag()
print("Salidas:", SALIDAS)
print("OPENAI_API_KEY definida:", clave_disponible())


Salidas: /home/jovyan/work/spark-apps/notebooks/salidas/rag
OPENAI_API_KEY definida: True


## 1. Preparación del conjunto de pruebas

Se utiliza un conjunto de preguntas con referencias documentales previamente definidas para evaluar la recuperación. Se incluyen consultas de diferentes temas y niveles de complejidad, así como preguntas cuya respuesta puede no encontrarse en el corpus.


In [2]:
meta = resumen_conjunto()
print(meta)
assert not meta["choques_reservados"], meta["choques_reservados"]
assert meta["incluye_rag_d_01_11"] == [f"RAG-D-{i:02d}" for i in range(1, 12)]
inv = pd.DataFrame([
    {
        "id": q["id"],
        "tema": q["tema"],
        "tipo": q["tipo"],
        "cobertura_esperada": q["cobertura_esperada"],
        "n_gold": len(q["relevantes"]),
        "pregunta": q["pregunta"],
    }
    for q in PREGUNTAS_AMPLIADAS
])
display(inv)
print(inv.groupby(["tema", "cobertura_esperada"]).size().unstack(fill_value=0))


{'n_preguntas': 40, 'n_con_gold': 29, 'n_sin_gold': 11, 'cobertura_esperada': {'suficiente': 29, 'no_cubierta': 8, 'parcial': 3}, 'tipo': {'definicion': 17, 'repartida': 5, 'fuera': 8, 'parafraseo': 2, 'excepcion': 1, 'cifra': 4, 'parcial': 3}, 'tema': {'minima_cuantia': 9, 'requisitos_habilitantes': 7, 'diccionario_secop': 5, 'c1_isolation_forest': 11, 'fuera_de_corpus': 8}, 'choques_reservados': [], 'incluye_rag_d_01_11': ['RAG-D-01', 'RAG-D-02', 'RAG-D-03', 'RAG-D-04', 'RAG-D-05', 'RAG-D-06', 'RAG-D-07', 'RAG-D-08', 'RAG-D-09', 'RAG-D-10', 'RAG-D-11']}


,id,tema,tipo,cobertura_esperada,n_gold,pregunta
0,RAG-D-01,minima_cuantia,definicion,suficiente,2,¿Qué es la modalidad de mínima cuantía?
1,RAG-D-02,minima_cuantia,definicion,suficiente,1,¿Cómo se determina el valor de la mínima cuantía?
2,RAG-D-03,requisitos_habilitantes,definicion,suficiente,3,¿Qué son los requisitos habilitantes y para qu...
3,RAG-D-04,requisitos_habilitantes,definicion,suficiente,3,¿Qué es el Registro Único de Proponentes (RUP)?
4,RAG-D-05,diccionario_secop,definicion,suficiente,1,¿Qué significa el campo Nit Entidad en el dicc...
5,RAG-D-06,c1_isolation_forest,repartida,suficiente,1,¿Qué variables usa la matriz C1 de Isolation F...
6,RAG-D-07,c1_isolation_forest,definicion,suficiente,1,¿Una puntuación alta de Isolation Forest es un...
7,RAG-D-08,c1_isolation_forest,repartida,suficiente,2,¿Cuántos contratos aptos se puntuaron en c1_if...
8,RAG-D-09,fuera_de_corpus,fuera,no_cubierta,0,¿Cuál es el procedimiento sancionatorio de Col...
9,RAG-D-10,fuera_de_corpus,fuera,no_cubierta,0,¿Qué dice el SISBÉN sobre la contratación públ...


cobertura_esperada       no_cubierta  parcial  suficiente
tema                                                     
c1_isolation_forest                0        1          10
diccionario_secop                  0        1           4
fuera_de_corpus                    8        0           0
minima_cuantia                     0        1           8
requisitos_habilitantes            0        0           7


## 2. Carga de representaciones vectoriales

Se cargan los embeddings previamente generados para representar los fragmentos documentales. Se verifica su disponibilidad y consistencia sin volver a calcularlos.

In [3]:
pack = cargar_indice_embeddings(reutilizar=True)
assert pack["desde_cache"], "Este notebook no debe regenerar embeddings."
vec = pack["indice"]["vectores"]
print("desde_cache:", pack["desde_cache"])
print("shape:", vec.shape)
print("modelo:", pack["modelo"])
assert vec.shape == (234, 1536)
indice = pack["indice"]


desde_cache: True
shape: (234, 1536)
modelo: text-embedding-3-small


## 3. Comparación de estrategias de recuperación

Se comparan tres estrategias: búsqueda por similitud coseno, recuperación con fragmentos adyacentes y combinación de búsqueda semántica y léxica mediante RRF. Los resultados se evalúan utilizando el mismo conjunto de preguntas.


In [4]:
ev = evaluar_ampliada(indice, top_k=5, umbral=0.30, salidas=SALIDAS)
met = ev["metricas"]
print("v3 conservado (no recalcula si ya existe):", ev.get("v3_conservado"))
print("UTC", met.get("generado_utc"))
print("n_preguntas", met["n_preguntas"])
print("embeddings", met["embeddings_shape"], "reutilizados", met["embeddings_reutilizados"])
print("hibrido:", met["hibrido"])
print("aviso:", met["aviso_muestra"])
comp = pd.DataFrame([
    {k: e[k] for k in [
        "id", "nombre", "n_con_gold", "recall_at_5", "mrr", "aciertos",
        "aciertos_cobertura", "falsos_positivos", "falsos_negativos",
        "fuera_rechazados_evidencia", "fuera_falsos_positivos",
        "fuera_rechazados_umbral_0_30",
    ]}
    for e in met["estrategias"]
])
display(comp)


v3 conservado (no recalcula si ya existe): True
UTC 2026-11-02T21:49:53Z
n_preguntas 40
embeddings [234, 1536] reutilizados True
hibrido: RRF k=60 de coseno y BM25 (k1=1.2, b=0.75); no se ajustó a las preguntas.
aviso: Cuarenta preguntas de desarrollo no generalizan el rechazo ni el ranking. No se afirma que el detector de evidencia ni la fusión RRF funcionen fuera de RAG-D.


,id,nombre,n_con_gold,recall_at_5,mrr,aciertos,aciertos_cobertura,falsos_positivos,falsos_negativos,fuera_rechazados_evidencia,fuera_falsos_positivos,fuera_rechazados_umbral_0_30
0,A,coseno_top5,29,0.8908,0.7672,28,27,0,3,7,0,0
1,B,coseno_top5_adyacentes,29,0.9655,0.7787,29,28,0,3,7,0,0
2,C,hibrido_rrf_top5,29,0.8621,0.7138,27,28,0,3,7,0,0


## 4. Análisis de recuperación por pregunta

Se revisan los resultados obtenidos para cada pregunta, identificando los documentos recuperados y su correspondencia con las referencias esperadas. Se presta especial atención a los casos que presentan dificultades de recuperación.

In [5]:
cols = [
    "id", "tema", "tipo", "cobertura_esperada", "n_gold",
    "a_recall", "b_recall_ctx", "c_recall",
    "a_rango", "b_rango", "c_rango",
    "a_cobertura", "b_cobertura", "c_cobertura",
    "a_confusion", "b_confusion", "c_confusion",
]
display(ev["resumen"][cols])
print("Foco problemas previos:")
display(pd.DataFrame(met["foco_d01_d02_d06_d08"]))


,id,tema,tipo,cobertura_esperada,n_gold,a_recall,b_recall_ctx,c_recall,a_rango,b_rango,c_rango,a_cobertura,b_cobertura,c_cobertura,a_confusion,b_confusion,c_confusion
0,RAG-D-01,minima_cuantia,definicion,suficiente,2,0.5000,0.5,0.5000,1.0,1.0,1.0,suficiente,suficiente,suficiente,acierto,acierto,acierto
1,RAG-D-02,minima_cuantia,definicion,suficiente,1,1.0000,1.0,1.0000,2.0,2.0,1.0,suficiente,suficiente,suficiente,acierto,acierto,acierto
2,RAG-D-03,requisitos_habilitantes,definicion,suficiente,3,0.6667,1.0,0.3333,2.0,2.0,4.0,suficiente,suficiente,suficiente,acierto,acierto,acierto
3,RAG-D-04,requisitos_habilitantes,definicion,suficiente,3,1.0000,1.0,1.0000,1.0,1.0,1.0,suficiente,suficiente,suficiente,acierto,acierto,acierto
4,RAG-D-05,diccionario_secop,definicion,suficiente,1,1.0000,1.0,1.0000,1.0,1.0,3.0,suficiente,suficiente,suficiente,acierto,acierto,acierto
5,RAG-D-06,c1_isolation_forest,repartida,suficiente,1,0.0000,1.0,1.0000,NaN,3.0,4.0,parcial,suficiente,suficiente,subestimacion,acierto,acierto
6,RAG-D-07,c1_isolation_forest,definicion,suficiente,1,1.0000,1.0,1.0000,1.0,1.0,1.0,suficiente,suficiente,suficiente,acierto,acierto,acierto
7,RAG-D-08,c1_isolation_forest,repartida,suficiente,2,0.5000,1.0,0.0000,1.0,1.0,NaN,parcial,parcial,parcial,subestimacion,subestimacion,subestimacion
8,RAG-D-09,fuera_de_corpus,fuera,no_cubierta,0,NaN,NaN,NaN,NaN,NaN,NaN,no_cubierta,no_cubierta,no_cubierta,acierto,acierto,acierto
9,RAG-D-10,fuera_de_corpus,fuera,no_cubierta,0,NaN,NaN,NaN,NaN,NaN,NaN,no_cubierta,no_cubierta,no_cubierta,acierto,acierto,acierto


Foco problemas previos:


,id,pregunta,a_recall,a_rango,a_top1_id,b_recall_ctx,b_rango,b_top1_id,c_recall,c_rango,c_top1_id,a_cobertura,b_cobertura,c_cobertura
0,RAG-D-01,¿Qué es la modalidad de mínima cuantía?,0.5,1.0,cce-minima-cuantia-2024-p005-009-9e9f94b4,0.5,1.0,cce-minima-cuantia-2024-p005-009-9e9f94b4,0.5,1.0,cce-minima-cuantia-2024-p005-009-9e9f94b4,suficiente,suficiente,suficiente
1,RAG-D-02,¿Cómo se determina el valor de la mínima cuantía?,1.0,2.0,cce-minima-cuantia-2024-p005-009-9e9f94b4,1.0,2.0,cce-minima-cuantia-2024-p005-009-9e9f94b4,1.0,1.0,cce-minima-cuantia-2024-p005-010-612d9d78,suficiente,suficiente,suficiente
2,RAG-D-06,¿Qué variables usa la matriz C1 de Isolation F...,0.0,NaN,ficha-c1-if-v1-p001-002-63fbcd8d,1.0,3.0,ficha-c1-if-v1-p001-002-63fbcd8d,1.0,4.0,ficha-c1-if-v1-p001-002-63fbcd8d,parcial,suficiente,suficiente
3,RAG-D-08,¿Cuántos contratos aptos se puntuaron en c1_if...,0.5,1.0,ficha-c1-if-v1-p001-008-2a27f6af,1.0,1.0,ficha-c1-if-v1-p001-008-2a27f6af,0.0,NaN,ficha-c1-if-v1-p001-002-63fbcd8d,parcial,parcial,parcial


## 5. Evaluación de la cobertura documental

Se analiza si los fragmentos recuperados proporcionan evidencia suficiente para responder las preguntas. Se comparan las clasificaciones de cobertura con las respuestas esperadas, considerando también los casos de información parcial o ausente.


In [6]:
for e in met["estrategias"]:
    print("=== Estrategia", e["id"], e["nombre"], "===")
    print(
        "cobertura predicha suficiente/parcial/no_cubierta:",
        e["n_suficiente_pred"], e["n_parcial_pred"], e["n_no_cubierta_pred"],
    )
    print(
        "FP / FN / FP suave / FN suave:",
        e["falsos_positivos"], e["falsos_negativos"],
        e["falsos_positivos_suaves"], e["falsos_negativos_suaves"],
    )
    print(
        "fuera evidencia rechazados / FP / umbral 0.30:",
        e["fuera_rechazados_evidencia"], e["fuera_falsos_positivos"],
        e["fuera_rechazados_umbral_0_30"],
    )
    print("por tema", e["por_tema"])
    print("por tipo", e["por_tipo"])

pref = "b"
cm = pd.crosstab(
    ev["resumen"]["cobertura_esperada"],
    ev["resumen"][f"{pref}_cobertura"],
    rownames=["esperado"],
    colnames=["observado B"],
)
display(cm)
errs = ev["resumen"][ev["resumen"][f"{pref}_confusion"] != "acierto"][
    ["id", "tipo", "pregunta", "cobertura_esperada", f"{pref}_cobertura", f"{pref}_confusion", f"{pref}_razon"]
]
print("Desacuerdos de evidencia (estrategia B):")
display(errs)


=== Estrategia A coseno_top5 ===
cobertura predicha suficiente/parcial/no_cubierta: 21 7 12
FP / FN / FP suave / FN suave: 0 3 1 2
fuera evidencia rechazados / FP / umbral 0.30: 7 0 0
por tema [{'tema': 'c1_isolation_forest', 'n': 10, 'recall_at_5': 0.85}, {'tema': 'diccionario_secop', 'n': 4, 'recall_at_5': 1.0}, {'tema': 'minima_cuantia', 'n': 8, 'recall_at_5': 0.8333}, {'tema': 'requisitos_habilitantes', 'n': 7, 'recall_at_5': 0.9524}]
por tipo [{'tipo': 'cifra', 'n': 4, 'recall_at_5': 1.0}, {'tipo': 'definicion', 'n': 17, 'recall_at_5': 0.9314}, {'tipo': 'excepcion', 'n': 1, 'recall_at_5': 1.0}, {'tipo': 'parafraseo', 'n': 2, 'recall_at_5': 1.0}, {'tipo': 'repartida', 'n': 5, 'recall_at_5': 0.6}]
=== Estrategia B coseno_top5_adyacentes ===
cobertura predicha suficiente/parcial/no_cubierta: 22 6 12
FP / FN / FP suave / FN suave: 0 3 1 2
fuera evidencia rechazados / FP / umbral 0.30: 7 0 0
por tema [{'tema': 'c1_isolation_forest', 'n': 10, 'recall_at_5': 1.0}, {'tema': 'diccionario_s

observado B,no_cubierta,parcial,suficiente
esperado,,,
no_cubierta,7,1,0
parcial,2,0,1
suficiente,3,5,21


Desacuerdos de evidencia (estrategia B):


,id,tipo,pregunta,cobertura_esperada,b_cobertura,b_confusion,b_razon
7,RAG-D-08,repartida,¿Cuántos contratos aptos se puntuaron en c1_if...,suficiente,parcial,subestimacion,Cobertura léxica parcial; no hay término disti...
11,RAG-D-12,parafraseo,¿En qué consiste el procedimiento de selección...,suficiente,parcial,subestimacion,Los hits no incluyen los términos distintivos ...
16,RAG-D-17,cifra,¿Durante cuánto tiempo debe publicarse el info...,suficiente,parcial,subestimacion,Los hits no incluyen los términos distintivos ...
17,RAG-D-18,parcial,¿Cuál es el valor en pesos de la mínima cuantí...,parcial,no_cubierta,falso_negativo_suave,El corpus no menciona: bogota.
18,RAG-D-19,definicion,¿La matriz C1 incluye el plazo de ejecución o ...,suficiente,parcial,subestimacion,Solo parte de los términos distintivos aparece...
19,RAG-D-20,cifra,¿Cuántos árboles (n_estimators) usa Isolation ...,suficiente,no_cubierta,falso_negativo,El corpus no menciona: arboles.
24,RAG-D-25,parcial,¿Cuántos contratos de Bogotá se puntuaron en c...,parcial,no_cubierta,falso_negativo_suave,El corpus no menciona: bogota.
25,RAG-D-26,repartida,¿Cuántos contratos se excluyeron de c1_if_v1 y...,suficiente,no_cubierta,falso_negativo,El corpus no menciona: excluyeron.
29,RAG-D-30,parcial,¿Qué significa el campo valor del contrato en ...,parcial,suficiente,sobreestimacion,Los hits contienen los términos distintivos de...
30,RAG-D-31,parafraseo,¿Los requisitos habilitantes se califican con ...,suficiente,no_cubierta,falso_negativo,El corpus no menciona: califican.


## 6. Verificación de resultados de recuperación

Se comprueba la generación de los archivos de evaluación y se revisan las métricas obtenidas para las diferentes estrategias.

In [7]:
print("Salidas v3:")
for p in sorted(SALIDAS.glob("*v3*")):
    print(" ", p.name, p.stat().st_size)
print("gold_ampliado.json", (SALIDAS / "gold_ampliado.json").stat().st_size)


Salidas v3:
  cobertura_evidencia_v3.csv 8567
  comparacion_estrategias_v3.csv 2192
  comparacion_estrategias_v3.json 4758
  metricas_recuperacion_v3.json 9268
  pruebas_recuperacion_detalle_v3.csv 574999
  pruebas_recuperacion_v3.csv 38890
gold_ampliado.json 19500


## 7. Evaluación del alcance de la recuperación

Se revisa la estrategia seleccionada incorporando criterios de alcance documental cuando las preguntas hacen referencia a modalidades o procedimientos específicos. Se analizan sus resultados sin modificar las evaluaciones anteriores


In [8]:
from rag.evaluar_ampliada import evaluar_cierre
from rag.pruebas_ampliadas import METRICAS_12C_B
from rag.pruebas_holdout import PREGUNTAS_HOLDOUT, comprobar_holdout

assert not comprobar_holdout(), comprobar_holdout()
print("Holdout:", [(q["id"], q["cobertura_esperada"], q["regla"]) for q in PREGUNTAS_HOLDOUT])
print("Antes 12c B (congelado):", METRICAS_12C_B)
cierre = evaluar_cierre(indice, top_k=5, umbral=0.30, salidas=SALIDAS)
met4 = cierre["metricas"]
print("v3 intactas:", cierre["v3_intactas"])
print("UTC v4", met4["generado_utc"])
print(met4["aviso"])


Holdout: [('RAG-H-01', 'suficiente', 'alcance_modalidad'), ('RAG-H-02', 'suficiente', 'termino_tecnico'), ('RAG-H-03', 'parcial', 'calificador_instancia'), ('RAG-H-04', 'suficiente', 'campo_exacto'), ('RAG-H-05', 'no_cubierta', 'version_producto'), ('RAG-H-06', 'parcial', 'calificador_instancia'), ('RAG-H-07', 'suficiente', 'alcance_modalidad'), ('RAG-H-08', 'suficiente', 'regla_general_sin_modalidad')]
Antes 12c B (congelado): {'recall_at_5': 0.9655, 'mrr': 0.7787, 'aciertos': 29, 'n_con_gold': 29, 'aciertos_cobertura': 28, 'falsos_positivos': 0, 'falsos_positivos_suaves': 1, 'falsos_negativos': 3, 'parcial_pred_ok': 0, 'fuera_rechazados_evidencia': 7, 'fuera_n': 8, 'n_preguntas': 40}


/home/jovyan/work/spark-apps/src/rag/evidencia.py:103: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return bool(col.str.contains(pat, regex=True).any())
/home/jovyan/work/spark-apps/src/rag/evidencia.py:103: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return bool(col.str.contains(pat, regex=True).any())
/home/jovyan/work/spark-apps/src/rag/evidencia.py:103: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return bool(col.str.contains(pat, regex=True).any())
/home/jovyan/work/spark-apps/src/rag/evidencia.py:103: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return bool(col.str.contains(pat, regex=True).any())
/home/jovyan/work/spark-apps

v3 intactas: True
UTC v4 2026-11-02T22:16:32Z
Las 40 preguntas RAG-D se usaron para observar fallos y no son validación independiente. RAG-H-* no participó en el ajuste.


## 8. Comparación de cobertura (muestra 40 vs holdout)

Se comparan los resultados obtenidos con el conjunto utilizado durante el desarrollo y con preguntas reservadas para validación. Esta comparación permite examinar la capacidad de recuperación sobre consultas que no participaron en los ajustes previos.

In [9]:
antes = met4["antes_12c_estrategia_B"]
desp = met4["despues_muestra_40"]
hold = met4["holdout_8"]
comp = pd.DataFrame([
    {"conjunto": "12c B antes (40)", **{k: antes.get(k) for k in [
        "recall_at_5", "mrr", "aciertos_cobertura", "falsos_positivos",
        "falsos_negativos", "parcial_pred_ok", "fuera_rechazados_evidencia",
    ]}},
    {"conjunto": "B+alcance+evidencia (40, no independiente)", **{k: desp.get(k) for k in [
        "recall_at_5", "mrr", "aciertos_cobertura", "falsos_positivos",
        "falsos_negativos", "parcial_pred_ok", "fuera_rechazados_evidencia",
    ]}},
    {"conjunto": "holdout RAG-H (8, no usadas en el ajuste)", **{k: hold.get(k) for k in [
        "recall_at_5", "mrr", "aciertos_cobertura", "falsos_positivos",
        "falsos_negativos", "parcial_pred_ok", "fuera_rechazados_evidencia",
    ]}},
])
display(comp)
print("Exactitud cobertura 40:", desp.get("exactitud_cobertura"), "holdout:", hold.get("exactitud_cobertura"))
print("Matriz 40:", desp["matriz_confusion"])
print("Matriz holdout:", hold["matriz_confusion"])
print("Por tipo 40:", desp["por_tipo"])
print("Excepción D-14:")
display(pd.DataFrame(met4["excepcion_d14"]))
print("Holdout excepción/control:")
display(pd.DataFrame(met4["excepcion_holdout"]))


,conjunto,recall_at_5,mrr,aciertos_cobertura,falsos_positivos,falsos_negativos,parcial_pred_ok,fuera_rechazados_evidencia
0,12c B antes (40),0.9655,0.7787,28,0,3,0,7
1,"B+alcance+evidencia (40, no independiente)",0.9828,0.7960,40,0,0,3,8
2,"holdout RAG-H (8, no usadas en el ajuste)",0.9333,0.7067,8,0,0,2,1


Exactitud cobertura 40: 1.0 holdout: 1.0
Matriz 40: {'suficiente': {'suficiente': 29, 'parcial': 0, 'no_cubierta': 0}, 'parcial': {'suficiente': 0, 'parcial': 3, 'no_cubierta': 0}, 'no_cubierta': {'suficiente': 0, 'parcial': 0, 'no_cubierta': 8}}
Matriz holdout: {'suficiente': {'suficiente': 5, 'parcial': 0, 'no_cubierta': 0}, 'parcial': {'suficiente': 0, 'parcial': 2, 'no_cubierta': 0}, 'no_cubierta': {'suficiente': 0, 'parcial': 0, 'no_cubierta': 1}}
Por tipo 40: [{'tipo': 'cifra', 'n': 4, 'aciertos_cobertura': 4, 'recall_at_5': 1.0}, {'tipo': 'definicion', 'n': 17, 'aciertos_cobertura': 17, 'recall_at_5': 0.9706}, {'tipo': 'excepcion', 'n': 1, 'aciertos_cobertura': 1, 'recall_at_5': 1.0}, {'tipo': 'fuera', 'n': 8, 'aciertos_cobertura': 8, 'recall_at_5': None}, {'tipo': 'parafraseo', 'n': 2, 'aciertos_cobertura': 2, 'recall_at_5': 1.0}, {'tipo': 'parcial', 'n': 3, 'aciertos_cobertura': 3, 'recall_at_5': None}, {'tipo': 'repartida', 'n': 5, 'aciertos_cobertura': 5, 'recall_at_5': 1.0}

,id,tema,tipo,pregunta,en_corpus,doc_id,cobertura_esperada,n_gold,gold,justificacion_gold,...,top1_similitud,recall,recall_ctx,rango,cobertura,razon,confusion,umbral_0_30,n_hits,doc_esperado_ok
0,RAG-D-14,minima_cuantia,repartida,¿Los oferentes deben acreditar el RUP en un pr...,True,cce-minima-cuantia-2024,suficiente,2,cce-minima-cuantia-2024-p010-027-e5bf81f3;cce-...,p010-027: los oferentes no están obligados a a...,...,0.693,1.0,1.0,1.0,suficiente,La pregunta no tiene términos raros; los hits ...,acierto,False,13,True


Holdout excepción/control:


,id,tema,tipo,pregunta,en_corpus,doc_id,cobertura_esperada,n_gold,gold,justificacion_gold,...,top1_similitud,recall,recall_ctx,rango,cobertura,razon,confusion,umbral_0_30,n_hits,doc_esperado_ok
0,RAG-H-01,minima_cuantia,excepcion,¿Hay que presentar el certificado del RUP si e...,True,cce-minima-cuantia-2024,suficiente,2,cce-minima-cuantia-2024-p010-027-e5bf81f3;cce-...,Misma excepción que el manual de mínima cuantí...,...,0.6443,1.0,1.0,1.0,suficiente,La pregunta no tiene términos raros; los hits ...,acierto,False,15,True
1,RAG-H-08,requisitos_habilitantes,definicion,¿El RUP es plena prueba para las entidades som...,True,cce-requisitos-habilitantes-v3-2023,suficiente,3,cce-requisitos-habilitantes-v3-2023-p009-029-8...,Regla general del manual de requisitos. No nom...,...,0.7032,1.0,1.0,3.0,suficiente,Los hits contienen los términos distintivos de...,acierto,False,9,True


In [10]:
cols = ["id", "tipo", "cobertura_esperada", "cobertura", "confusion", "alcance", "top1_doc", "recall_ctx", "razon"]
print("Desacuerdos en la muestra 40:")
display(cierre["muestra"][cierre["muestra"]["confusion"] != "acierto"][cols])
print("Holdout completo:")
display(cierre["holdout"][cols])
import json
orig12c = json.loads((SALIDAS / "metricas_recuperacion_v3.json").read_text(encoding="utf-8"))
print("12c B recall", orig12c["estrategias"][1]["recall_at_5"])
assert orig12c["estrategias"][1]["recall_at_5"] == 0.9655


Desacuerdos en la muestra 40:


,id,tipo,cobertura_esperada,cobertura,confusion,alcance,top1_doc,recall_ctx,razon


Holdout completo:


,id,tipo,cobertura_esperada,cobertura,confusion,alcance,top1_doc,recall_ctx,razon
0,RAG-H-01,excepcion,suficiente,suficiente,acierto,cce-minima-cuantia-2024,cce-minima-cuantia-2024,1.0000,La pregunta no tiene términos raros; los hits ...
1,RAG-H-02,cifra,suficiente,suficiente,acierto,ficha-c1-if-v1,ficha-c1-if-v1,1.0000,Los hits contienen los términos distintivos de...
2,RAG-H-03,parcial,parcial,parcial,acierto,ficha-c1-if-v1,ficha-c1-if-v1,NaN,"Hay evidencia del tema, pero no del recorte pe..."
3,RAG-H-04,definicion,suficiente,suficiente,acierto,diccionario-datos-abiertos-secop,diccionario-datos-abiertos-secop,1.0000,Los hits contienen los términos distintivos de...
4,RAG-H-05,fuera,no_cubierta,no_cubierta,acierto,cce-requisitos-habilitantes-v3-2023,cce-requisitos-habilitantes-v3-2023,NaN,El corpus no cubre la versión o plataforma: se...
5,RAG-H-06,parcial,parcial,parcial,acierto,None,cce-minima-cuantia-2024,NaN,"Hay evidencia del tema, pero no del recorte pe..."
6,RAG-H-07,definicion,suficiente,suficiente,acierto,cce-minima-cuantia-2024,cce-minima-cuantia-2024,0.6667,La pregunta no tiene términos raros; los hits ...
7,RAG-H-08,definicion,suficiente,suficiente,acierto,cce-requisitos-habilitantes-v3-2023,cce-requisitos-habilitantes-v3-2023,1.0000,Los hits contienen los términos distintivos de...


12c B recall 0.9655


## Integración de la recuperación documental con el chat

Se evalúa la integración de la estrategia de recuperación seleccionada con el servicio de conversación del asistente. Se verifica el comportamiento de las consultas documentales mediante la API del chat, utilizando la configuración de recuperación previamente establecida y sin modificar los embeddings existentes..


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

aqui = Path.cwd().resolve()
RAIZ = next(
    (
        c
        for c in (aqui, *aqui.parents)
        if (c / "src" / "rag").is_dir() and (c / "notebooks").is_dir()
    ),
    None,
)
if RAIZ is None:
    raise FileNotFoundError("Ejecute el notebook desde el repositorio secop-tfm.")
sys.path.insert(0, str(RAIZ / "src"))

from nlq_eval.particion import IDS_RESERVADOS
from rag.evaluar_chat import evaluar_chat_documental
from rag.preguntas_chat import PREGUNTAS_CHAT
from rag.rutas import ruta_salidas_rag

SALIDAS = ruta_salidas_rag()
print("preguntas", len(PREGUNTAS_CHAT))
print("reservados tocados", [p["id"] for p in PREGUNTAS_CHAT if p["id"] in IDS_RESERVADOS])
print("salidas", SALIDAS)
assert not any(p["id"] in IDS_RESERVADOS for p in PREGUNTAS_CHAT)
for nombre in (
    "metricas_recuperacion_v4.json",
    "embeddings_text-embedding-3-small.npz",
):
    assert (SALIDAS / nombre).is_file(), nombre


In [ ]:
salida = evaluar_chat_documental(salidas=SALIDAS)
print("bloqueo", salida["bloqueo"])
if salida["bloqueo"]:
    print(json.dumps(salida["metricas"], ensure_ascii=False, indent=2))
else:
    met = salida["metricas"]
    print(json.dumps({k: met[k] for k in [
        "generado_utc", "n", "n_ok", "exactitud", "recuperacion", "citas",
        "latencia_media_s", "latencia_max_s", "ids_ok", "ids_error"
    ]}, ensure_ascii=False, indent=2))
    display(salida["resultados"][[
        "id", "caso", "intent_observado", "cobertura_observada", "ok", "segundos", "texto"
    ]])


In [ ]:
assert (SALIDAS / "metricas_recuperacion_v4.json").is_file()
v4 = json.loads((SALIDAS / "metricas_recuperacion_v4.json").read_text(encoding="utf-8"))
print("v4 estrategia", v4.get("estrategia"), "shape", v4.get("embeddings_shape"))
from nlq_eval.rutas import ruta_salidas_nlq
nlq = ruta_salidas_nlq()
for f in (
    "metricas_lenguaje_ciudadano_v1.json",
    "metricas_lenguaje_ciudadano_v2.json",
    "etiquetas_referencia_v1.csv",
):
    p = nlq / f
    print(f, p.is_file())


## Comparación del asistente con y sin RAG

Se compara el comportamiento del modelo GPT-4o-mini al responder preguntas de contratación pública en dos condiciones: sin información documental recuperada y utilizando el sistema RAG. El objetivo es evaluar cómo influye la incorporación de evidencia documental en la calidad y fundamentación de las respuestas.


In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

aqui = Path.cwd().resolve()
RAIZ = next(
    (
        c
        for c in (aqui, *aqui.parents)
        if (c / "src" / "rag").is_dir() and (c / "notebooks").is_dir()
    ),
    None,
)
if RAIZ is None:
    raise FileNotFoundError("Ejecute el notebook desde el repositorio secop-tfm.")
sys.path.insert(0, str(RAIZ / "src"))

from rag.comparacion import MODELO, TEMPERATURA, ejecutar_comparacion, prompt_sin_rag
from rag.preguntas_comparacion import PREGUNTAS_COMPARACION, validar_preguntas
from rag.rutas import ruta_salidas_rag
from nlq_eval.particion import IDS_RESERVADOS

SALIDAS = ruta_salidas_rag()
validar_preguntas()
print("preguntas", len(PREGUNTAS_COMPARACION))
print("familias", dict(Counter(p["familia"] for p in PREGUNTAS_COMPARACION)))
print("modelo", MODELO, "temperatura", TEMPERATURA)
print("reservados tocados", [p["id"] for p in PREGUNTAS_COMPARACION if p["id"] in IDS_RESERVADOS])
assert "FRAGMENTO" not in prompt_sin_rag("prueba")
assert "corpus v1 del TFM" in prompt_sin_rag("prueba")
for f in (
    "metricas_recuperacion_v4.json",
    "embeddings_text-embedding-3-small.npz",
    "embeddings_meta.json",
):
    assert (SALIDAS / f).is_file(), f


preguntas 12
familias {'respondible': 5, 'parcial': 2, 'fuera_corpus': 2, 'minima_rup': 1, 'vigencia': 2}
modelo gpt-4o-mini temperatura 0
reservados tocados []


## Protocolo de evaluación

Se establece un conjunto común de preguntas y condiciones de ejecución para realizar la comparación. Se consideran criterios relacionados con la exactitud de la información, el reconocimiento de limitaciones, la utilización de referencias documentales y la correspondencia de las respuestas con las fuentes recuperadas.

| Condición | Qué se llama | Fragmentos |
|---|---|---|
| A sin RAG | gpt-4o-mini, temperature=0, prompt JSON sin hits | ninguno |
| B con RAG | POST /api/chat + recuperador B congelado | top-5 + adyacentes + alcance |


In [2]:
proto = pd.DataFrame(PREGUNTAS_COMPARACION)[
    ["id", "familia", "tema", "pregunta", "cobertura_esperada", "doc_id_esperado", "criterio"]
]
display(proto)


,id,familia,tema,pregunta,cobertura_esperada,doc_id_esperado,criterio
0,COMP-01,respondible,minima_cuantia,¿Qué es la modalidad de mínima cuantía?,suficiente,cce-minima-cuantia-2024,El manual 2024 define la modalidad como convoc...
1,COMP-02,respondible,rup,¿Qué es el Registro Único de Proponentes (RUP)?,suficiente,cce-requisitos-habilitantes-v3-2023,El manual de requisitos habilitantes v3 identi...
2,COMP-03,respondible,c1_isolation_forest,¿Cuántos árboles (n_estimators) usa Isolation ...,suficiente,ficha-c1-if-v1,La ficha publicada fija n_estimators = 200. El...
3,COMP-04,respondible,c1_isolation_forest,¿Qué variables usa la matriz C1 de Isolation F...,suficiente,ficha-c1-if-v1,"C1 usa log1p_valor, log_valor_vs_modalidad y d..."
4,COMP-05,respondible,c1_isolation_forest,¿Una puntuación alta de Isolation Forest demue...,suficiente,ficha-c1-if-v1,La ficha afirma que una puntuación alta no es ...
5,COMP-06,parcial,minima_cuantia,¿Cuál es el valor en pesos de la mínima cuantí...,parcial,cce-minima-cuantia-2024,"El manual trae el 10 % de la menor cuantía, no..."
6,COMP-07,parcial,c1_isolation_forest,¿Cuántos contratos de Bogotá se puntuaron en c...,parcial,ficha-c1-if-v1,La ficha da 82.457 aptos nacionales de 2026-09...
7,COMP-08,fuera_corpus,fuera,¿Qué es el SISBÉN IV?,no_cubierta,None,SISBÉN no está en el corpus v1. Con RAG debe d...
8,COMP-09,fuera_corpus,excluido,¿Cuál es el procedimiento sancionatorio de Col...,no_cubierta,None,La guía sancionatoria está excluida del corpus...
9,COMP-10,minima_rup,rup,¿Los oferentes deben acreditar el RUP en un pr...,suficiente,cce-minima-cuantia-2024,Excepción del manual de mínima cuantía: no es ...


In [3]:
salida = ejecutar_comparacion(salidas=SALIDAS)
print("bloqueo", salida.get("bloqueo"))
if salida.get("bloqueo"):
    print(json.dumps(salida, ensure_ascii=False, indent=2)[:2000])
else:
    met = salida["metricas"]
    print(json.dumps({
        "generado_utc": met["generado_utc"],
        "n": met["n_preguntas"],
        "distribucion": met["distribucion"],
        "corpus": met["corpus"],
        "por_condicion": met["por_condicion"],
    }, ensure_ascii=False, indent=2))


bloqueo False
{
  "generado_utc": "2026-11-03T01:29:15Z",
  "n": 12,
  "distribucion": {
    "respondible": 5,
    "parcial": 2,
    "fuera_corpus": 2,
    "vigencia": 2,
    "minima_rup": 1
  },
  "corpus": {
    "modelo_embedding": "text-embedding-3-small",
    "n_vectores": 234,
    "dimension": 1536,
    "huella": "ea2bd163922584f786094ec05a93769cbdb77e0800ecbaa63a6e6ed7d8934dc9",
    "embeddings_utc": "2026-11-02T20:06:31Z",
    "estrategia_congelada": "B_coseno_top5_adyacentes_alcance",
    "v4_utc": "2026-11-02T22:16:32Z"
  },
  "por_condicion": {
    "sin_rag": {
      "n": 12,
      "exactitud_factual": 0.1667,
      "respaldo": 1.0,
      "reconoce_insuficiencia": 0.5,
      "citas_validas": 1.0,
      "citas_pertinentes": 1.0,
      "inventa_citas": 0.0,
      "parcial_ok": 0.0,
      "n_sin_respaldo_medio": null,
      "latencia_media_s": 1.528,
      "latencia_max_s": 2.618,
      "ids_ok_factual": [
        "COMP-05",
        "COMP-10"
      ],
      "ids_error_factual": 

In [4]:
if salida.get("bloqueo"):
    print("Sin comparación.")
else:
    res = salida["resultados"]
    cols = [
        "id", "familia", "condicion", "factual_ok", "respaldo_ok",
        "reconoce_insuficiencia", "citas_validas", "inventa_citas",
        "segundos", "n_referencias",
    ]
    display(res[cols])
    fig_dir = SALIDAS / "figuras"
    fig_dir.mkdir(parents=True, exist_ok=True)
    por = salida["metricas"]["por_condicion"]
    metricas_plot = ["exactitud_factual", "respaldo", "reconoce_insuficiencia", "citas_validas"]
    x = range(len(metricas_plot))
    a = [por["sin_rag"].get(m) or 0 for m in metricas_plot]
    b = [por["con_rag"].get(m) or 0 for m in metricas_plot]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar([i - 0.2 for i in x], a, width=0.4, label="sin RAG")
    ax.bar([i + 0.2 for i in x], b, width=0.4, label="con RAG")
    ax.set_xticks(list(x))
    ax.set_xticklabels(metricas_plot, rotation=20, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Proporción")
    ax.set_title("Comparación automática (12 preguntas de desarrollo)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(fig_dir / "comparacion_con_sin_rag_v1.png", dpi=120)
    plt.show()


,id,familia,condicion,factual_ok,respaldo_ok,reconoce_insuficiencia,citas_validas,inventa_citas,segundos,n_referencias
0,COMP-01,respondible,sin_rag,False,True,False,True,False,2.618,0
1,COMP-01,respondible,con_rag,True,True,False,True,False,2.881,2
2,COMP-02,respondible,sin_rag,False,True,False,True,False,1.697,0
3,COMP-02,respondible,con_rag,True,True,False,True,False,3.349,3
4,COMP-03,respondible,sin_rag,False,True,False,True,False,1.398,0
5,COMP-03,respondible,con_rag,True,True,False,True,False,2.782,1
6,COMP-04,respondible,sin_rag,False,True,False,True,False,1.475,0
7,COMP-04,respondible,con_rag,True,True,False,True,False,2.319,2
8,COMP-05,respondible,sin_rag,True,True,False,True,False,1.524,0
9,COMP-05,respondible,con_rag,True,True,False,True,False,1.971,1


In [5]:
if salida.get("bloqueo"):
    print("Sin ejemplos.")
else:
    res = salida["resultados"]
    print("=== Diferencias factuales ===")
    for eid in [p["id"] for p in PREGUNTAS_COMPARACION]:
        a = res[(res["id"] == eid) & (res["condicion"] == "sin_rag")].iloc[0]
        b = res[(res["id"] == eid) & (res["condicion"] == "con_rag")].iloc[0]
        if bool(a["factual_ok"]) != bool(b["factual_ok"]):
            print(eid, "sin", bool(a["factual_ok"]), "con", bool(b["factual_ok"]))
            print(" A:", str(a["texto"])[:280].replace("\n", " "))
            print(" B:", str(b["texto"])[:280].replace("\n", " "))
            print()
    conservados = [
        "metricas_recuperacion_v4.json",
        "metricas_chat_rag_v1.json",
    ]
    for f in conservados:
        print("conservado", f, (SALIDAS / f).is_file())
    from nlq_eval.rutas import ruta_salidas_nlq
    nlq = ruta_salidas_nlq()
    print("nlq v1", (nlq / "metricas_lenguaje_ciudadano_v1.json").is_file())
    print("nlq v2", (nlq / "metricas_lenguaje_ciudadano_v2.json").is_file())


=== Diferencias factuales ===
COMP-01 sin False con True
 A: La modalidad de mínima cuantía es un procedimiento de contratación pública en Colombia que se utiliza para la adquisición de bienes y servicios cuyo valor no supera un monto establecido por la ley. Este tipo de contratación busca simplificar los procesos y facilitar la participac
 B: La mínima cuantía es una modalidad de selección en la que la entidad estatal realiza una convocatoria pública para recibir ofertas de bienes, obras o servicios cuyo valor no excede el diez por ciento de su menor cuantía.  Los manuales del corpus no acreditan por sí mismos la vige

COMP-02 sin False con True
 A: El Registro Único de Proponentes (RUP) es un sistema que agrupa la información de los proponentes que desean participar en procesos de contratación pública en Colombia. Su objetivo es facilitar la identificación y verificación de los proponentes, asegurando que cumplan con los re
 B: El Registro Único de Proponentes (RUP) es un documento q